# 06 — Inference: Translate New Podcast Transcripts

**What this notebook does and why it exists**

This is the lightest notebook in the project — its only job is to take a German podcast transcript and produce an English translation. It handles chunking of long transcripts into sentence-sized segments, translates each chunk in order, and saves the result to a text file.

This notebook is designed to run on a free Colab CPU tier. It has no training dependencies — you do not need Unsloth, bitsandbytes, or TRL. If you want to run it locally on your Mac using the MLX model, see the note at the bottom of `05_mlx_conversion.ipynb`.

**Inputs:** A `.txt` file with your German transcript (one segment per line, or a continuous block of text)  
**Output:** A `.txt` file with the English translation, in the same segment order

In [ ]:
# ── Minimal dependencies ─────────────────────────────────────────────────────
# We deliberately keep this notebook lean so it runs on a free CPU tier.
!pip install -q \
    transformers==4.45.0 \
    peft==0.13.2 \
    torch==2.4.0 \
    sentencepiece==0.2.0

print('Dependencies installed.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR    = '/content/drive/MyDrive/podcast_translation'
GRPO_DIR    = os.path.join(BASE_DIR, 'grpo_adapter', 'final')
OUTPUT_DIR  = os.path.join(BASE_DIR, 'translations')
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE_MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
PROMPT_TEMPLATE = (
    'Translate the following German podcast transcript to natural English.\n\n'
    'German: {german}\n\n'
    'Translation:'
)

print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
# ── Configuration — edit these ───────────────────────────────────────────────

# Path to your German transcript file (one segment per line, or continuous text)
# Upload your file to Colab or point to a Drive path.
TRANSCRIPT_PATH = '/content/drive/MyDrive/my_podcast_transcript.txt'

# Output file name (saved to OUTPUT_DIR)
OUTPUT_FILENAME = 'translated_transcript.txt'

# Maximum tokens per segment. Segments longer than this are split at sentence
# boundaries. Shorter = more API calls but more accurate chunking.
MAX_SEGMENT_TOKENS = 80

# Generation parameters
MAX_NEW_TOKENS  = 150
DO_SAMPLE       = False   # Greedy decoding for consistency (change to True for variety)

print('Configuration set.')

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print(f'Loading base model: {BASE_MODEL_NAME}')
tokeniser = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokeniser.pad_token is None:
    tokeniser.pad_token = tokeniser.eos_token

# Load in float32 for CPU inference (bfloat16 is not supported on all CPU environments)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float32,
    trust_remote_code=True
)

if os.path.isdir(GRPO_DIR):
    print(f'Loading LoRA adapter from: {GRPO_DIR}')
    model = PeftModel.from_pretrained(model, GRPO_DIR)
    model = model.merge_and_unload()
    print('Adapter merged.')
else:
    print(f'No adapter found at {GRPO_DIR}. Running base model only.')

model.eval()
print('Model ready.')

In [ ]:
# ── Transcript loading and chunking ──────────────────────────────────────────
import re

def split_into_segments(text: str, max_tokens: int) -> list:
    """Split text into segments of at most max_tokens tokens.
    
    Strategy:
    1. Split on newlines first (respects the original segment boundaries)
    2. For lines that are too long, split at sentence-ending punctuation
    3. As a last resort, split at max_tokens with a hard cut
    """
    # First, split on newlines
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    
    segments = []
    for line in lines:
        token_count = len(tokeniser.encode(line))
        if token_count <= max_tokens:
            segments.append(line)
        else:
            # Split on German sentence boundaries
            # German sentences end in . ! ? followed by space and capital letter
            sentences = re.split(r'(?<=[.!?])\s+(?=[A-ZÄÖÜ])', line)
            current = ''
            for sent in sentences:
                candidate = (current + ' ' + sent).strip()
                if len(tokeniser.encode(candidate)) <= max_tokens:
                    current = candidate
                else:
                    if current:
                        segments.append(current)
                    current = sent
            if current:
                segments.append(current)
    
    return [s for s in segments if s.strip()]

# Load transcript
if os.path.exists(TRANSCRIPT_PATH):
    with open(TRANSCRIPT_PATH, encoding='utf-8') as f:
        raw_text = f.read()
    segments = split_into_segments(raw_text, MAX_SEGMENT_TOKENS)
    print(f'Transcript loaded: {len(raw_text):,} characters → {len(segments)} segments')
else:
    # Demo with sample text if no transcript is provided
    print(f'File not found: {TRANSCRIPT_PATH}')
    print('Using demo text for demonstration.')
    demo = (
        'Herzlich willkommen zu unserem Podcast.\n'
        'Heute sprechen wir über künstliche Intelligenz und ihre Auswirkungen auf die Gesellschaft.\n'
        'Das ist ein sehr wichtiges Thema, das uns alle betrifft.\n'
        'Ich finde, dass wir sehr offen darüber reden müssen.'
    )
    segments = split_into_segments(demo, MAX_SEGMENT_TOKENS)
    print(f'Demo text → {len(segments)} segments')

In [ ]:
# ── Translate all segments ────────────────────────────────────────────────────
from tqdm.auto import tqdm

@torch.no_grad()
def translate_segment(german: str) -> str:
    prompt = PROMPT_TEMPLATE.format(german=german) + ' '
    inputs = tokeniser(
        prompt, return_tensors='pt', truncation=True, max_length=256
    ).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        temperature=0.8 if DO_SAMPLE else 1.0,
        pad_token_id=tokeniser.eos_token_id,
    )
    gen = tokeniser.decode(
        out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True
    ).strip()
    # Remove anything after the first sentence-ending punctuation that
    # is followed by a newline (the model sometimes leaks the next prompt)
    gen = gen.split('\n')[0].strip()
    return gen

translations = []
for i, seg in enumerate(tqdm(segments, desc='Translating')):
    en = translate_segment(seg)
    translations.append(en)

print(f'\nTranslation complete: {len(translations)} segments')

In [ ]:
# ── Preview first 10 segments ─────────────────────────────────────────────────
print('=== PREVIEW (first 10 segments) ===')
for i, (de, en) in enumerate(zip(segments[:10], translations[:10]), 1):
    print(f'[{i}] DE: {de}')
    print(f'    EN: {en}')
    print()

In [ ]:
# ── Save translated transcript ─────────────────────────────────────────────────
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

with open(output_path, 'w', encoding='utf-8') as f:
    for en in translations:
        f.write(en + '\n')

print(f'Translated transcript saved to: {output_path}')
print(f'Total segments: {len(translations)}')
print(f'Total words (approx): {sum(len(t.split()) for t in translations):,}')

---
## Tips for Better Results

**If the translation truncates mid-sentence:** Increase `MAX_NEW_TOKENS`. For very long German sentences (80+ tokens) that produce English translations over 150 tokens, this can happen.

**If the model produces repetitive output:** Enable sampling (`DO_SAMPLE = True`) with `temperature=0.7`. This adds controlled randomness that often breaks repetition loops.

**If segments are being split at the wrong places:** Preprocess your transcript to have one German sentence per line before passing it to this notebook. The chunker works best when line breaks correspond to natural sentence boundaries.

**Running on your Mac locally (MLX version):**  
After downloading the MLX archive from notebook 05, replace the model loading cell with:
```python
from mlx_lm import load, generate
model, tokenizer = load('/path/to/mlx_model')
def translate_segment(german):
    prompt = PROMPT_TEMPLATE.format(german=german) + ' '
    return generate(model, tokenizer, prompt=prompt, max_tokens=150).strip()
```